# 05 — Neural Tier (DistilBERT fine-tune)

Fine-tunes `distilbert-base-uncased` on the train split via the HuggingFace `Trainer`. CPU is technically possible but very slow; a Colab T4 / Apple-Silicon MPS / any CUDA GPU is strongly recommended.

For speed, this notebook fine-tunes on a **20K subset** of the train split by default — sufficient for a baseline fine-tuned number to beat the classical tier. Bump `MAX_TRAIN` to use the full split.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

import os
import numpy as np
import polars as pl
import torch

from utils import ARTIFACTS_DIR, save_metrics, set_seed
set_seed()

MODEL_NAME = "distilbert-base-uncased"
MAX_TRAIN  = 20_000
MAX_VAL    = 2_000
MAX_TEST   = 5_000
NUM_EPOCHS = 2
BATCH_SIZE = 16
LR         = 5e-5

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

In [ ]:
train = pl.read_parquet(ARTIFACTS_DIR / "split_train.parquet").to_pandas().sample(MAX_TRAIN, random_state=42)
val   = pl.read_parquet(ARTIFACTS_DIR / "split_val.parquet").to_pandas().sample(MAX_VAL,   random_state=42)
test  = pl.read_parquet(ARTIFACTS_DIR / "split_test.parquet").to_pandas().sample(MAX_TEST,  random_state=42)
print(train.shape, val.shape, test.shape)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import evaluate as hf_eval

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(df):
    ds = Dataset.from_pandas(df[["review", "label"]].reset_index(drop=True))
    return ds.map(
        lambda b: tok(b["review"], truncation=True, padding="max_length", max_length=192),
        batched=True,
        remove_columns=["review"],
    )

ds_train = to_ds(train)
ds_val   = to_ds(val)
ds_test  = to_ds(test)

In [ ]:
acc_metric = hf_eval.load("accuracy")
f1_metric  = hf_eval.load("f1")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=p.label_ids)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=p.label_ids, average="macro")["f1"],
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir=str(ARTIFACTS_DIR / "distilbert_runs"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    report_to="none",
    fp16=(device == "cuda"),
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
metrics = trainer.evaluate(ds_test)
print(metrics)
save_metrics("neural_distilbert", {**metrics, "model": MODEL_NAME, "n_train": MAX_TRAIN})
trainer.save_model(str(ARTIFACTS_DIR / "distilbert_final"))
tok.save_pretrained(str(ARTIFACTS_DIR / "distilbert_final"))

## Findings to record

- Compare DistilBERT test-set accuracy / macro-F1 (above) against the classical-ML best (notebook 04). The lift is the headline neural-tier result.
- Training curve / loss is logged to `data/artifacts/distilbert_runs/`. Export a screenshot or re-plot for the LaTeX figure.